# Quantify tracked roots

Run this after `track.ipynb`, in the same session. It reads every tip-coordinate CSV in
`/app/results/tip_coordinates/` and writes one table per seed to `/app/results/quantification/`:
displacement from the root's own centerline, circumnutation peaks and period, and elongation
along the path. The tables are meant to be loaded and analysed elsewhere.

**Scale (`px_per_mm`) is a property of the rig** — camera, lens and working distance — so it has
to be right for the robot that took these images. 28.6 px/mm was measured on the College Station
rig from the 76.2 mm box pitch. Once a run's `run_config.json` records it, it is picked up
automatically and these arguments can be dropped.

Ported from the lab's R analysis; see `reference/r_analysis/README.md` for what changed.

In [ ]:
%%capture
import sys
sys.path.append('/app/')
from src.myutilities import quantify

In [ ]:
data = quantify.quantify_dir(
    tip_dir="/app/results/tip_coordinates",
    out_dir="/app/results/quantification",
    px_per_mm=28.6,     # College Station; from the 76.2 mm box pitch
    interval_min=15,    # imaging cycle; read from run_config.json when present
    span=0.2,           # centerline smoothing (R: loess span; 0.6 for very slow growth)
    curl_trim=True,     # drop the tail where the root curls back on itself
)
data.head()

In [ ]:
# per-seed summary of what was written
if len(data):
    data.groupby(["box", "seed"]).agg(
        frames=("frame", "size"),
        hours=("growth_h", "max"),
        swings=("is_max", "sum"),
        median_period_h=("period_h", "median"),
        amplitude_mm=("amplitude_mm", lambda a: a.abs().max()),
        grew_mm=("arc_length_mm", "max"),
    )